In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
import warnings
import time
from pathlib import Path
import os
from datetime import datetime
from joblib import Parallel, delayed
import pickle
warnings.filterwarnings('ignore')

# Set professional plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9

class GroundwaterPredictorMultiRatio:
    """Enhanced groundwater prediction analysis with multiple train/test ratios and parallel processing."""
    
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.df_filtered = None
        self.all_results = {}
        self.feature_cols = []
        self.target_col = 'Depth to water (m)'
        self.models = {
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Decision Tree': DecisionTreeRegressor(random_state=42)
        }
        
        # Test ratios to evaluate
        self.test_ratios = [
            (10, 90), (20, 80), (30, 70), (40, 60), (50, 50),
            (60, 40), (70, 30), (80, 20), (90, 10)
        ]
        
        # Timing variables
        self.start_time = None
        self.timing_log = []
        
        # Create main output directory
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.main_output_dir = Path(f"groundwater_ratio_analysis_{timestamp}")
        self.main_output_dir.mkdir(exist_ok=True)
        
        print(f"📁 Results will be saved to: {self.main_output_dir}")
        
    def log_time(self, step_name):
        """Log timing information for each step."""
        current_time = time.time()
        if self.start_time is None:
            self.start_time = current_time
            elapsed = 0
        else:
            elapsed = current_time - self.start_time
        
        self.timing_log.append({
            'step': step_name,
            'elapsed_minutes': elapsed / 60,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })
        
        print(f"⏱️  {step_name} completed at {datetime.now().strftime('%H:%M:%S')} "
              f"(Total elapsed: {elapsed/60:.2f} minutes)")
    
    def mean_absolute_percentage_error(self, y_true, y_pred):
        """Calculate Mean Absolute Percentage Error (MAPE)"""
        epsilon = 1e-10
        return np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100
    
    def load_and_preprocess_data(self):
        """Load and preprocess the dataset."""
        print("🌊 Chile Groundwater Depth Prediction Analysis - Multi Ratio Testing")
        print("📊 Testing Multiple Train/Test Split Ratios with Parallel Processing")
        print("=" * 80)
        print(f"🚀 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("\n📊 Loading and preprocessing dataset...")
        
        self.log_time("Analysis Started")
        
        try:
            if self.file_path.endswith('.csv'):
                self.df = pd.read_csv(self.file_path)
            else:
                self.df = pd.read_excel(self.file_path)
            print(f"✅ Dataset loaded successfully! Shape: {self.df.shape}")
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            # Create sample data for demonstration
            print("Creating sample data for demonstration...")
            np.random.seed(42)
            n_samples = 1000
            
            # Create sample data with multiple records per well
            n_wells = 200
            records_per_well = np.random.poisson(5, n_wells) + 1
            
            well_data = []
            for i in range(n_wells):
                well_code = f'Well_{i:04d}'
                n_records = records_per_well[i]
                
                base_lon = np.random.uniform(-75, -65)
                base_lat = np.random.uniform(-35, -25)
                base_elevation = np.random.normal(500, 200)
                
                for j in range(n_records):
                    date_value = pd.Timestamp('2020-01-01') + pd.Timedelta(days=j*30 + np.random.randint(0, 30))
                    
                    well_data.append({
                        'Code': well_code,
                        'Date_String': date_value,
                        'Depth to water (m)': np.random.exponential(10),
                        'Longitude_GCS_WGS_1984': base_lon + np.random.normal(0, 0.01),
                        'Latitude_GCS_WGS_1984': base_lat + np.random.normal(0, 0.01),
                        'elevation_NASADEM': base_elevation + np.random.normal(0, 50),
                        'slope_NASADEM': np.random.exponential(5),
                        'terraclim_pr_value': np.random.exponential(100),
                        'terraclim_tmmn_value': np.random.normal(100, 50),
                        'terraclim_tmmx_value': np.random.normal(200, 50),
                        'Basin': np.random.choice(['Basin_A', 'Basin_B', 'Basin_C']),
                        'Status': np.random.choice(['Active', 'Inactive', np.nan], p=[0.3, 0.3, 0.4])
                    })
            
            self.df = pd.DataFrame(well_data)
        
        self.log_time("Data Loading")
        
        # Filter data
        print(f"Original dataset shape: {self.df.shape}")
        self.df_filtered = self.df[self.df['Status'].isnull()].copy()
        print(f"Filtered dataset shape (Status is blank): {self.df_filtered.shape}")
        
        if len(self.df_filtered) == 0:
            print("❌ No rows with blank Status found! Using all data for demonstration.")
            self.df_filtered = self.df.copy()
        
        # Remove rows where target variable is missing
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered.dropna(subset=[self.target_col])
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with missing target values")
        
        # Prepare features
        self._prepare_features()
        self.log_time("Data Preprocessing")
        
    def _prepare_features(self):
        """Prepare feature matrix and target variable."""
        # Identify numeric and categorical columns
        numeric_cols = self.df_filtered.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df_filtered.select_dtypes(include=['object']).columns.tolist()
        
        # Remove target and irrelevant columns
        if self.target_col in numeric_cols:
            numeric_cols.remove(self.target_col)
        
        columns_to_remove = ['Status', 'Code', 'Date_String']
        for col in columns_to_remove:
            if col in categorical_cols:
                categorical_cols.remove(col)
            if col in numeric_cols:
                numeric_cols.remove(col)
        
        # Handle missing values in numeric columns
        for col in numeric_cols:
            if self.df_filtered[col].isnull().sum() > 0:
                self.df_filtered[col].fillna(self.df_filtered[col].median(), inplace=True)
        
        # Encode categorical variables
        for col in categorical_cols:
            if col in self.df_filtered.columns:
                if self.df_filtered[col].isnull().sum() > 0:
                    self.df_filtered[col].fillna(self.df_filtered[col].mode()[0], inplace=True)
                
                le = LabelEncoder()
                self.df_filtered[col + '_encoded'] = le.fit_transform(self.df_filtered[col].astype(str))
        
        # Create feature columns list
        encoded_categorical_cols = [col + '_encoded' for col in categorical_cols if col in self.df_filtered.columns]
        self.feature_cols = numeric_cols + encoded_categorical_cols
        print(f"Final feature set: {len(self.feature_cols)} features")
    
    def create_improved_temporal_well_split(self, X, y, train_ratio=0.7, min_records_per_well=9):
        """Improved temporal split strategy for groundwater level prediction."""
        
        if 'Code' not in self.df_filtered.columns:
            print("Warning: 'Code' column not found. Using random split instead.")
            return train_test_split(X, y, test_size=1-train_ratio, random_state=42)
        
        # Create a date column if it doesn't exist
        if 'Date_String' not in self.df_filtered.columns:
            print("Creating synthetic date ordering based on data sequence...")
            self.df_filtered['Date_String'] = self.df_filtered.groupby('Code').cumcount()
        
        train_indices = []
        test_indices = []
        wells_with_insufficient_data = []
        
        # Get unique wells
        unique_wells = self.df_filtered['Code'].unique()
        
        for well in unique_wells:
            well_mask = self.df_filtered['Code'] == well
            well_data = self.df_filtered[well_mask].copy()
            
            well_data_sorted = well_data.sort_values('Date_String')
            well_indices = well_data_sorted.index.tolist()
            
            n_records = len(well_indices)
            
            if n_records < min_records_per_well:
                wells_with_insufficient_data.append(well)
                train_indices.extend(well_indices)
                continue
            
            n_train = max(1, int(n_records * train_ratio))
            n_train = min(n_train, n_records - 1)
            
            train_indices.extend(well_indices[:n_train])
            test_indices.extend(well_indices[n_train:])
        
        # Convert to boolean masks
        train_mask = self.df_filtered.index.isin(train_indices)
        test_mask = self.df_filtered.index.isin(test_indices)
        
        return X[train_mask], X[test_mask], y[train_mask], y[test_mask]
    
    def _calculate_metrics(self, y_true, y_pred):
        """Calculate performance metrics."""
        return {
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'mae': mean_absolute_error(y_true, y_pred),
            'r2': r2_score(y_true, y_pred),
            'mape': self.mean_absolute_percentage_error(y_true, y_pred)
        }
    
    def analyze_single_ratio(self, train_pct, test_pct):
        """Analyze a single train/test ratio - designed for parallel execution."""
        ratio_name = f"{train_pct:02d}_{test_pct:02d}"
        train_ratio = train_pct / 100.0
        
        print(f"🔄 Processing ratio {train_pct}/{test_pct}...")
        
        # Create directory for this ratio
        ratio_dir = self.main_output_dir / f"ratio_{ratio_name}"
        ratio_dir.mkdir(exist_ok=True)
        
        # Prepare data
        X = self.df_filtered[self.feature_cols].values
        y = self.df_filtered[self.target_col].values
        
        # Apply temporal split
        X_train, X_test, y_train, y_test = self.create_improved_temporal_well_split(
            X, y, train_ratio=train_ratio, min_records_per_well=9
        )
        
        # Train and evaluate models
        ratio_results = {}
        
        for name, model in self.models.items():
            model_start_time = time.time()
            
            # Create a fresh model instance to avoid conflicts in parallel processing
            if name == 'Extra Trees':
                fresh_model = ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=1)
            elif name == 'Random Forest':
                fresh_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=1)
            else:
                fresh_model = DecisionTreeRegressor(random_state=42)
            
            # Train model
            fresh_model.fit(X_train, y_train)
            
            # Make predictions
            y_pred_train = fresh_model.predict(X_train)
            y_pred_test = fresh_model.predict(X_test)
            
            # Calculate metrics
            train_metrics = self._calculate_metrics(y_train, y_pred_train)
            test_metrics = self._calculate_metrics(y_test, y_pred_test)
            
            # Cross-validation
            cv_scores = cross_val_score(fresh_model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
            cv_rmse = np.sqrt(-cv_scores.mean())
            
            model_time = (time.time() - model_start_time) / 60
            
            ratio_results[name] = {
                'model': fresh_model,
                'train_metrics': train_metrics,
                'test_metrics': test_metrics,
                'cv_rmse': cv_rmse,
                'predictions': y_pred_test,
                'actual_test': y_test,
                'training_time_minutes': model_time,
                'train_size': len(X_train),
                'test_size': len(X_test)
            }
        
        # Create visualizations for this ratio
        self._create_ratio_visualizations(ratio_results, ratio_dir, ratio_name)
        
        # Save results
        with open(ratio_dir / 'results.pkl', 'wb') as f:
            pickle.dump(ratio_results, f)
        
        print(f"✅ Completed ratio {train_pct}/{test_pct}")
        
        return ratio_name, ratio_results
    
    def run_parallel_analysis(self, n_jobs=-1):
        """Run analysis for all ratios using parallel processing."""
        print(f"\n{'='*80}")
        print(f"🚀 Starting Parallel Analysis with {len(self.test_ratios)} different train/test ratios")
        print(f"{'='*80}")
        
        # Determine number of jobs
        if n_jobs == -1:
            n_jobs = min(len(self.test_ratios), os.cpu_count())
        
        print(f"🔧 Using {n_jobs} parallel processes")
        
        # Run parallel analysis
        start_time = time.time()
        
        # Use joblib for parallel processing
        results = Parallel(n_jobs=n_jobs, verbose=1)(
            delayed(self.analyze_single_ratio)(train_pct, test_pct) 
            for train_pct, test_pct in self.test_ratios
        )
        
        # Collect results
        for ratio_name, ratio_results in results:
            self.all_results[ratio_name] = ratio_results
        
        parallel_time = (time.time() - start_time) / 60
        print(f"⚡ Parallel analysis completed in {parallel_time:.2f} minutes")
        
        self.log_time("Parallel Analysis Complete")
        
        # Create summary analysis
        self._create_summary_analysis()
        
        # Create Excel report
        self._create_excel_report()
        
        # Print final summary
        self._print_final_summary()
        
        self.log_time("Complete Analysis Finished")
    
    def _create_ratio_visualizations(self, ratio_results, ratio_dir, ratio_name):
        """Create visualizations for a specific ratio."""
        
        # Model performance comparison for this ratio
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle(f'Model...{ratio_name.replace("_", "/")}', 
                     fontsize=16, fontweight='bold')
        
        model_names = list(ratio_results.keys())
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        
        # Test MAPE comparison
        test_mape = [ratio_results[name]['test_metrics']['mape'] for name in model_names]
        bars = axes[0, 0].bar(model_names, test_mape, color=colors)
        axes[0, 0].set_ylabel('Test MAPE (%)')
        axes[0, 0].set_title('Test MAPE Comparison')
        axes[0, 0].tick_params(axis='x', rotation=45)
        
        for bar, value in zip(bars, test_mape):
            axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(test_mape)*0.01,
                           f'{value:.2f}%', ha='center', va='bottom', fontweight='bold')
        
        # Test R² comparison
        test_r2 = [ratio_results[name]['test_metrics']['r2'] for name in model_names]
        bars = axes[0, 1].bar(model_names, test_r2, color=colors)
        axes[0, 1].set_ylabel('Test R² Score')
        axes[0, 1].set_title('Test R² Comparison')
        axes[0, 1].tick_params(axis='x', rotation=45)
        
        for bar, value in zip(bars, test_r2):
            axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(test_r2)*0.01,
                           f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
        
        # Predicted vs Actual for best model (lowest MAPE)
        best_model = min(model_names, key=lambda x: ratio_results[x]['test_metrics']['mape'])
        y_test = ratio_results[best_model]['actual_test']
        y_pred = ratio_results[best_model]['predictions']
        
        axes[1, 0].scatter(y_test, y_pred, alpha=0.6, color=colors[0], s=20)
        axes[1, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        axes[1, 0].set_xlabel('Actual Depth to Water (m)')
        axes[1, 0].set_ylabel('Predicted Depth to Water (m)')
        axes[1, 0].set_title(f'{best_model}: Predicted vs Actual')
        
        # Training vs Testing MAPE
        train_mape = [ratio_results[name]['train_metrics']['mape'] for name in model_names]
        x_pos = np.arange(len(model_names))
        width = 0.35
        
        axes[1, 1].bar(x_pos - width/2, train_mape, width, label='Train MAPE', color='lightblue', alpha=0.8)
        axes[1, 1].bar(x_pos + width/2, test_mape, width, label='Test MAPE', color='lightcoral', alpha=0.8)
        axes[1, 1].set_ylabel('MAPE (%)')
        axes[1, 1].set_title('Training vs Testing MAPE')
        axes[1, 1].set_xticks(x_pos)
        axes[1, 1].set_xticklabels(model_names, rotation=45)
        axes[1, 1].legend()
        
        plt.tight_layout()
        plt.savefig(ratio_dir / 'model_performance.png', bbox_inches='tight', facecolor='white')
        plt.close()  # Close to save memory
    
    def _create_summary_analysis(self):
        """Create comprehensive summary analysis across all ratios."""
        print("\n📊 Creating summary analysis across all ratios...")
        
        # Create summary comparison plots
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        fig.suptitle('Performance Comparison Across All Train/Test Ratios', fontsize=18, fontweight='bold')
        
        # Prepare data for plotting
        ratios = [f"{r[0]}/{r[1]}" for r in self.test_ratios]
        models = list(next(iter(self.all_results.values())).keys())
        
        # Test MAPE across ratios
        for i, model in enumerate(models):
            test_mapes = []
            for ratio_key in self.all_results.keys():
                test_mapes.append(self.all_results[ratio_key][model]['test_metrics']['mape'])
            
            axes[0, 0].plot(ratios, test_mapes, marker='o', label=model, linewidth=2, markersize=6)
        
        axes[0, 0].set_xlabel('Train/Test Ratio')
        axes[0, 0].set_ylabel('Test MAPE (%)')
        axes[0, 0].set_title('Test MAPE vs Train/Test Ratio')
        axes[0, 0].legend()
        axes[0, 0].tick_params(axis='x', rotation=45)
        axes[0, 0].grid(True, alpha=0.3)
        
        # Test R² across ratios
        for i, model in enumerate(models):
            test_r2s = []
            for ratio_key in self.all_results.keys():
                test_r2s.append(self.all_results[ratio_key][model]['test_metrics']['r2'])
            
            axes[0, 1].plot(ratios, test_r2s, marker='o', label=model, linewidth=2, markersize=6)
        
        axes[0, 1].set_xlabel('Train/Test Ratio')
        axes[0, 1].set_ylabel('Test R²')
        axes[0, 1].set_title('Test R² vs Train/Test Ratio')
        axes[0, 1].legend()
        axes[0, 1].tick_params(axis='x', rotation=45)
        axes[0, 1].grid(True, alpha=0.3)
        
        # Training time across ratios
        for i, model in enumerate(models):
            train_times = []
            for ratio_key in self.all_results.keys():
                train_times.append(self.all_results[ratio_key][model]['training_time_minutes'])
            
            axes[0, 2].plot(ratios, train_times, marker='o', label=model, linewidth=2, markersize=6)
        
        axes[0, 2].set_xlabel('Train/Test Ratio')
        axes[0, 2].set_ylabel('Training Time (minutes)')
        axes[0, 2].set_title('Training Time vs Train/Test Ratio')
        axes[0, 2].legend()
        axes[0, 2].tick_params(axis='x', rotation=45)
        axes[0, 2].grid(True, alpha=0.3)
        
        # Sample sizes across ratios
        train_sizes = []
        test_sizes = []
        for ratio_key in self.all_results.keys():
            # Use first model as they all have same sample sizes
            first_model = list(self.all_results[ratio_key].keys())[0]
            train_sizes.append(self.all_results[ratio_key][first_model]['train_size'])
            test_sizes.append(self.all_results[ratio_key][first_model]['test_size'])
        
        axes[1, 0].bar(ratios, train_sizes, alpha=0.7, label='Training Samples', color='lightblue')
        axes[1, 0].bar(ratios, test_sizes, bottom=train_sizes, alpha=0.7, label='Testing Samples', color='lightcoral')
        axes[1, 0].set_xlabel('Train/Test Ratio')
        axes[1, 0].set_ylabel('Number of Samples')
        axes[1, 0].set_title('Sample Size Distribution')
        axes[1, 0].legend()
        axes[1, 0].tick_params(axis='x', rotation=45)
        
        # Best model performance heatmap
        performance_matrix = np.zeros((len(models), len(ratios)))
        for i, model in enumerate(models):
            for j, ratio_key in enumerate(self.all_results.keys()):
                performance_matrix[i, j] = self.all_results[ratio_key][model]['test_metrics']['mape']
        
        im = axes[1, 1].imshow(performance_matrix, cmap='RdYlBu_r', aspect='auto')
        axes[1, 1].set_xticks(range(len(ratios)))
        axes[1, 1].set_xticklabels(ratios, rotation=45)
        axes[1, 1].set_yticks(range(len(models)))
        axes[1, 1].set_yticklabels(models)
        axes[1, 1].set_title('Test MAPE Heatmap')
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1, 1])
        cbar.set_label('Test MAPE (%)')
        
        # Add text annotations
        for i in range(len(models)):
            for j in range(len(ratios)):
                text = axes[1, 1].text(j, i, f'{performance_matrix[i, j]:.1f}',
                                     ha="center", va="center", color="white", fontweight='bold')
        
        # Overfitting analysis (Train MAPE - Test MAPE)
        for i, model in enumerate(models):
            overfitting_scores = []
            for ratio_key in self.all_results.keys():
                train_mape = self.all_results[ratio_key][model]['train_metrics']['mape']
                test_mape = self.all_results[ratio_key][model]['test_metrics']['mape']
                overfitting_scores.append(test_mape - train_mape)
            
            axes[1, 2].plot(ratios, overfitting_scores, marker='o', label=model, linewidth=2, markersize=6)
        
        axes[1, 2].set_xlabel('Train/Test Ratio')
        axes[1, 2].set_ylabel('Overfitting Score (Test MAPE - Train MAPE)')
        axes[1, 2].set_title('Overfitting Analysis')
        axes[1, 2].legend()
        axes[1, 2].tick_params(axis='x', rotation=45)
        axes[1, 2].grid(True, alpha=0.3)
        axes[1, 2].axhline(y=0, color='red', linestyle='--', alpha=0.7)
        
        plt.tight_layout()
        plt.savefig(self.main_output_dir / 'comprehensive_ratio_analysis.png', 
                   bbox_inches='tight', facecolor='white', dpi=300)
        plt.show()
    
    def _create_excel_report(self):
        """Create comprehensive Excel report with all results."""
        print("\n📊 Creating comprehensive Excel report...")
        
        excel_path = self.main_output_dir / 'comprehensive_results.xlsx'
        
        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
            
            # Summary sheet
            summary_data = []
            for ratio_key in self.all_results.keys():
                train_pct, test_pct = ratio_key.split('_')
                ratio_display = f"{train_pct}/{test_pct}"
                
                for model_name in self.all_results[ratio_key].keys():
                    result = self.all_results[ratio_key][model_name]
                    
                    summary_data.append({
                        'Train_Test_Ratio': ratio_display,
                        'Model': model_name,
                        'Train_Size': result['train_size'],
                        'Test_Size': result['test_size'],
                        'Train_MAPE': result['train_metrics']['mape'],
                        'Test_MAPE': result['test_metrics']['mape'],
                        'Train_R2': result['train_metrics']['r2'],
                        'Test_R2': result['test_metrics']['r2'],
                        'Train_RMSE': result['train_metrics']['rmse'],
                        'Test_RMSE': result['test_metrics']['rmse'],
                        'Train_MAE': result['train_metrics']['mae'],
                        'Test_MAE': result['test_metrics']['mae'],
                        'CV_RMSE': result['cv_rmse'],
                        'Training_Time_Minutes': result['training_time_minutes'],
                        'Overfitting_Score': result['test_metrics']['mape'] - result['train_metrics']['mape']
                    })
            
            summary_df = pd.DataFrame(summary_data)
            summary_df.to_excel(writer, sheet_name='Summary', index=False)
            
            # Best performance sheet
            best_results = []
            for ratio_key in self.all_results.keys():
                train_pct, test_pct = ratio_key.split('_')
                ratio_display = f"{train_pct}/{test_pct}"
                
                # Find best model for this ratio (lowest test MAPE)
                best_model = min(self.all_results[ratio_key].keys(), 
                               key=lambda x: self.all_results[ratio_key][x]['test_metrics']['mape'])
                
                result = self.all_results[ratio_key][best_model]
                
                best_results.append({
                    'Train_Test_Ratio': ratio_display,
                    'Best_Model': best_model,
                    'Test_MAPE': result['test_metrics']['mape'],
                    'Test_R2': result['test_metrics']['r2'],
                    'Test_RMSE': result['test_metrics']['rmse'],
                    'Test_MAE': result['test_metrics']['mae'],
                    'Training_Time_Minutes': result['training_time_minutes'],
                    'Train_Size': result['train_size'],
                    'Test_Size': result['test_size']
                })
            
            best_df = pd.DataFrame(best_results)
            best_df.to_excel(writer, sheet_name='Best_Per_Ratio', index=False)
            
            # Individual model sheets
            for model_name in models:
                model_data = []
                for ratio_key in self.all_results.keys():
                    train_pct, test_pct = ratio_key.split('_')
                    ratio_display = f"{train_pct}/{test_pct}"
                    
                    if model_name in self.all_results[ratio_key]:
                        result = self.all_results[ratio_key][model_name]
                        
                        model_data.append({
                            'Train_Test_Ratio': ratio_display,
                            'Train_Size': result['train_size'],
                            'Test_Size': result['test_size'],
                            'Train_MAPE': result['train_metrics']['mape'],
                            'Test_MAPE': result['test_metrics']['mape'],
                            'Train_R2': result['train_metrics']['r2'],
                            'Test_R2': result['test_metrics']['r2'],
                            'Train_RMSE': result['train_metrics']['rmse'],
                            'Test_RMSE': result['test_metrics']['rmse'],
                            'Train_MAE': result['train_metrics']['mae'],
                            'Test_MAE': result['test_metrics']['mae'],
                            'CV_RMSE': result['cv_rmse'],
                            'Training_Time_Minutes': result['training_time_minutes']
                        })
                
                model_df = pd.DataFrame(model_data)
                sheet_name = model_name.replace(' ', '_')[:31]  # Excel sheet name limit
                model_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            # Feature importance sheet (using one ratio as example)
            first_ratio_key = list(self.all_results.keys())[0]
            feature_importance_data = []
            
            for model_name in self.all_results[first_ratio_key].keys():
                model = self.all_results[first_ratio_key][model_name]['model']
                if hasattr(model, 'feature_importances_'):
                    importances = model.feature_importances_
                    for i, importance in enumerate(importances):
                        feature_importance_data.append({
                            'Model': model_name,
                            'Feature': self.feature_cols[i],
                            'Importance': importance
                        })
            
            if feature_importance_data:
                fi_df = pd.DataFrame(feature_importance_data)
                fi_df.to_excel(writer, sheet_name='Feature_Importance', index=False)
        
        print(f"✅ Excel report saved to: {excel_path}")
    
    def _print_final_summary(self):
        """Print comprehensive final summary."""
        print("\n" + "="*100)
        print("🎉 COMPREHENSIVE ANALYSIS COMPLETE - MULTI-RATIO EVALUATION")
        print("="*100)
        
        # Find overall best performing combination
        best_combo = None
        best_mape = float('inf')
        
        for ratio_key in self.all_results.keys():
            train_pct, test_pct = ratio_key.split('_')
            ratio_display = f"{train_pct}/{test_pct}"
            
            for model_name in self.all_results[ratio_key].keys():
                test_mape = self.all_results[ratio_key][model_name]['test_metrics']['mape']
                if test_mape < best_mape:
                    best_mape = test_mape
                    best_combo = (ratio_display, model_name)
        
        print(f"🏆 BEST OVERALL PERFORMANCE:")
        print(f"   Ratio: {best_combo[0]} | Model: {best_combo[1]} | Test MAPE: {best_mape:.2f}%")
        
        # Summary statistics
        print(f"\n📊 ANALYSIS SUMMARY:")
        print(f"   Total Ratios Tested: {len(self.test_ratios)}")
        print(f"   Models per Ratio: {len(self.models)}")
        print(f"   Total Model Runs: {len(self.test_ratios) * len(self.models)}")
        print(f"   Results Saved to: {self.main_output_dir}")
        
        # Performance trends
        print(f"\n📈 KEY INSIGHTS:")
        
        # Best ratio for each model
        for model_name in self.models.keys():
            best_ratio_for_model = None
            best_mape_for_model = float('inf')
            
            for ratio_key in self.all_results.keys():
                test_mape = self.all_results[ratio_key][model_name]['test_metrics']['mape']
                if test_mape < best_mape_for_model:
                    best_mape_for_model = test_mape
                    best_ratio_for_model = ratio_key.replace('_', '/')
            
            print(f"   • {model_name}: Best at {best_ratio_for_model} ratio (MAPE: {best_mape_for_model:.2f}%)")
        
        # Timing summary
        if self.timing_log:
            total_time = self.timing_log[-1]['elapsed_minutes']
            print(f"\n⏰ TOTAL RUNTIME: {total_time:.2f} minutes")
        
        print("="*100)

def main():
    """Main execution function."""
    # Update this path to your actual file location
    file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"
    
    # Initialize and run analysis
    predictor = GroundwaterPredictorMultiRatio(file_path)
    predictor.load_and_preprocess_data()
    
    # Run parallel analysis with all CPU cores (use n_jobs=4 to limit if needed)
    predictor.run_parallel_analysis(n_jobs=-1)

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
import warnings
import time
from pathlib import Path
import os
from datetime import datetime
from joblib import Parallel, delayed
import pickle
warnings.filterwarnings('ignore')

# Set professional plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9

class GroundwaterPredictorMultiRatio:
    """Enhanced groundwater prediction analysis with multiple train/test ratios and parallel processing."""

    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.df_filtered = None
        self.all_results = {}
        self.feature_cols = []
        self.target_col = 'Depth to water (m)'
        self.models = {
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Decision Tree': DecisionTreeRegressor(random_state=42)
        }

        # Test ratios to evaluate
        self.test_ratios = [
            (10, 90), (20, 80), (30, 70), (40, 60), (50, 50),
            (60, 40), (70, 30), (80, 20), (90, 10)
        ]

        # Timing variables
        self.start_time = None
        self.timing_log = []

        # Create main output directory
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.main_output_dir = Path(f"groundwater_ratio_analysis_{timestamp}")
        self.main_output_dir.mkdir(exist_ok=True)

        print(f"📁 Results will be saved to: {self.main_output_dir}")

    def log_time(self, step_name):
        """Log timing information for each step."""
        current_time = time.time()
        if self.start_time is None:
            self.start_time = current_time
            elapsed = 0
        else:
            elapsed = current_time - self.start_time

        self.timing_log.append({
            'step': step_name,
            'elapsed_minutes': elapsed / 60,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })

        print(f"⏱️  {step_name} completed at {datetime.now().strftime('%H:%M:%S')} "
              f"(Total elapsed: {elapsed/60:.2f} minutes)")

    def mean_absolute_percentage_error(self, y_true, y_pred):
        """Calculate Mean Absolute Percentage Error (MAPE)"""
        epsilon = 1e-10
        # Ensure y_true and y_pred are numpy arrays for element-wise operations
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        return np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100

    def load_and_preprocess_data(self):
        """Load and preprocess the dataset."""
        print("🌊 Chile Groundwater Depth Prediction Analysis - Multi Ratio Testing")
        print("📊 Testing Multiple Train/Test Split Ratios with Parallel Processing")
        print("=" * 80)
        print(f"🚀 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("\n📊 Loading and preprocessing dataset...")

        self.log_time("Analysis Started")

        try:
            if self.file_path.endswith('.csv'):
                self.df = pd.read_csv(self.file_path)
            else:
                self.df = pd.read_excel(self.file_path)
            print(f"✅ Dataset loaded successfully! Shape: {self.df.shape}")
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            # Create sample data for demonstration
            print("Creating sample data for demonstration...")
            np.random.seed(42)
            n_samples = 1000

            # Create sample data with multiple records per well
            n_wells = 200
            records_per_well = np.random.poisson(5, n_wells) + 1

            well_data = []
            for i in range(n_wells):
                well_code = f'Well_{i:04d}'
                n_records = records_per_well[i]

                base_lon = np.random.uniform(-75, -65)
                base_lat = np.random.uniform(-35, -25)
                base_elevation = np.random.normal(500, 200)

                for j in range(n_records):
                    date_value = pd.Timestamp('2020-01-01') + pd.Timedelta(days=j*30 + np.random.randint(0, 30))

                    well_data.append({
                        'Code': well_code,
                        'Date_String': date_value,
                        'Depth to water (m)': np.random.exponential(10),
                        'Longitude_GCS_WGS_1984': base_lon + np.random.normal(0, 0.01),
                        'Latitude_GCS_WGS_1984': base_lat + np.random.normal(0, 0.01),
                        'elevation_NASADEM': base_elevation + np.random.normal(0, 50),
                        'slope_NASADEM': np.random.exponential(5),
                        'terraclim_pr_value': np.random.exponential(100),
                        'terraclim_tmmn_value': np.random.normal(100, 50),
                        'terraclim_tmmx_value': np.random.normal(200, 50),
                        'Basin': np.random.choice(['Basin_A', 'Basin_B', 'Basin_C']),
                        'Status': np.random.choice(['Active', 'Inactive', np.nan], p=[0.3, 0.3, 0.4])
                    })

            self.df = pd.DataFrame(well_data)

        self.log_time("Data Loading")

        # Filter data
        print(f"Original dataset shape: {self.df.shape}")
        self.df_filtered = self.df[self.df['Status'].isnull()].copy()
        print(f"Filtered dataset shape (Status is blank): {self.df_filtered.shape}")

        if len(self.df_filtered) == 0:
            print("❌ No rows with blank Status found! Using all data for demonstration.")
            self.df_filtered = self.df.copy()

        # Remove rows where target variable is missing
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered.dropna(subset=[self.target_col])
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with missing target values")

        # Prepare features
        self._prepare_features()
        self.log_time("Data Preprocessing")

    def _prepare_features(self):
        """Prepare feature matrix and target variable."""
        # Identify numeric and categorical columns
        numeric_cols = self.df_filtered.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df_filtered.select_dtypes(include=['object']).columns.tolist()

        # Remove target and irrelevant columns
        if self.target_col in numeric_cols:
            numeric_cols.remove(self.target_col)

        columns_to_remove = ['Status', 'Code', 'Date_String']
        for col in columns_to_remove:
            if col in categorical_cols:
                categorical_cols.remove(col)
            if col in numeric_cols:
                numeric_cols.remove(col)

        # Handle missing values in numeric columns
        for col in numeric_cols:
            if self.df_filtered[col].isnull().sum() > 0:
                self.df_filtered[col].fillna(self.df_filtered[col].median(), inplace=True)

        # Encode categorical variables
        for col in categorical_cols:
            if col in self.df_filtered.columns:
                if self.df_filtered[col].isnull().sum() > 0:
                    self.df_filtered[col].fillna(self.df_filtered[col].mode()[0], inplace=True)

                le = LabelEncoder()
                self.df_filtered[col + '_encoded'] = le.fit_transform(self.df_filtered[col].astype(str))

        # Create feature columns list
        encoded_categorical_cols = [col + '_encoded' for col in categorical_cols if col in self.df_filtered.columns]
        self.feature_cols = numeric_cols + encoded_categorical_cols
        print(f"Final feature set: {len(self.feature_cols)} features")

    def create_improved_temporal_well_split(self, X, y, train_ratio=0.7, min_records_per_well=9):
        """Improved temporal split strategy for groundwater level prediction."""

        if 'Code' not in self.df_filtered.columns:
            print("Warning: 'Code' column not found. Using random split instead.")
            return train_test_split(X, y, test_size=1-train_ratio, random_state=42)

        # Create a date column if it doesn't exist
        if 'Date_String' not in self.df_filtered.columns:
            print("Creating synthetic date ordering based on data sequence...")
            self.df_filtered['Date_String'] = self.df_filtered.groupby('Code').cumcount()
        else: # Ensure date column is sortable
            self.df_filtered['Date_String'] = pd.to_datetime(self.df_filtered['Date_String'], errors='coerce')


        train_indices = []
        test_indices = []
        wells_with_insufficient_data = []

        # Get unique wells
        unique_wells = self.df_filtered['Code'].unique()

        for well in unique_wells:
            well_mask = self.df_filtered['Code'] == well
            well_data = self.df_filtered[well_mask].copy()

            well_data_sorted = well_data.sort_values('Date_String')
            well_indices = well_data_sorted.index.tolist()

            n_records = len(well_indices)

            if n_records < min_records_per_well:
                wells_with_insufficient_data.append(well)
                train_indices.extend(well_indices)
                continue

            n_train = max(1, int(n_records * train_ratio))
            n_train = min(n_train, n_records - 1)

            train_indices.extend(well_indices[:n_train])
            test_indices.extend(well_indices[n_train:])

        # Convert to boolean masks
        train_mask = self.df_filtered.index.isin(train_indices)
        test_mask = self.df_filtered.index.isin(test_indices)

        return X[train_mask], X[test_mask], y[train_mask], y[test_mask]

    def _calculate_metrics(self, y_true, y_pred):
        """Calculate performance metrics."""
        return {
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'mae': mean_absolute_error(y_true, y_pred),
            'r2': r2_score(y_true, y_pred),
            'mape': self.mean_absolute_percentage_error(y_true, y_pred)
        }

    def analyze_single_ratio(self, train_pct, test_pct):
        """Analyze a single train/test ratio - designed for parallel execution."""
        ratio_name = f"{train_pct:02d}_{test_pct:02d}"
        train_ratio = train_pct / 100.0

        print(f"🔄 Processing ratio {train_pct}/{test_pct}...")

        # Create directory for this ratio
        ratio_dir = self.main_output_dir / f"ratio_{ratio_name}"
        ratio_dir.mkdir(exist_ok=True)

        # Prepare data
        X = self.df_filtered[self.feature_cols].values
        y = self.df_filtered[self.target_col].values

        # Apply temporal split
        X_train, X_test, y_train, y_test = self.create_improved_temporal_well_split(
            X, y, train_ratio=train_ratio, min_records_per_well=9
        )
        
        # Handle case where split results in empty test set
        if len(X_test) == 0:
            print(f"⚠️ Skipping ratio {train_pct}/{test_pct} due to empty test set after temporal split.")
            return ratio_name, {}


        # Train and evaluate models
        ratio_results = {}

        for name, model in self.models.items():
            model_start_time = time.time()

            # Create a fresh model instance to avoid conflicts in parallel processing
            if name == 'Extra Trees':
                fresh_model = ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=1)
            elif name == 'Random Forest':
                fresh_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=1)
            else:
                fresh_model = DecisionTreeRegressor(random_state=42)

            # Train model
            fresh_model.fit(X_train, y_train)

            # Make predictions
            y_pred_train = fresh_model.predict(X_train)
            y_pred_test = fresh_model.predict(X_test)

            # Calculate metrics
            train_metrics = self._calculate_metrics(y_train, y_pred_train)
            test_metrics = self._calculate_metrics(y_test, y_pred_test)

            # Cross-validation
            cv_scores = cross_val_score(fresh_model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
            cv_rmse = np.sqrt(-cv_scores.mean())

            model_time = (time.time() - model_start_time) / 60

            ratio_results[name] = {
                'model': fresh_model,
                'train_metrics': train_metrics,
                'test_metrics': test_metrics,
                'cv_rmse': cv_rmse,
                'predictions': y_pred_test,
                'actual_test': y_test,
                'training_time_minutes': model_time,
                'train_size': len(X_train),
                'test_size': len(X_test)
            }

        # Create visualizations for this ratio
        self._create_ratio_visualizations(ratio_results, ratio_dir, ratio_name)

        # Save results
        with open(ratio_dir / 'results.pkl', 'wb') as f:
            pickle.dump(ratio_results, f)

        print(f"✅ Completed ratio {train_pct}/{test_pct}")

        return ratio_name, ratio_results

    def run_parallel_analysis(self, n_jobs=-1):
        """Run analysis for all ratios using parallel processing."""
        print(f"\n{'='*80}")
        print(f"🚀 Starting Parallel Analysis with {len(self.test_ratios)} different train/test ratios")
        print(f"{'='*80}")

        # Determine number of jobs
        if n_jobs == -1:
            n_jobs = min(len(self.test_ratios), os.cpu_count())

        print(f"🔧 Using {n_jobs} parallel processes")

        # Run parallel analysis
        start_time = time.time()

        # Use joblib for parallel processing
        results = Parallel(n_jobs=n_jobs, verbose=1)(
            delayed(self.analyze_single_ratio)(train_pct, test_pct)
            for train_pct, test_pct in self.test_ratios
        )

        # Collect results, filtering out any empty ones
        for result in results:
            if result is not None and result[1]: # Check for non-empty results
                ratio_name, ratio_results = result
                self.all_results[ratio_name] = ratio_results

        parallel_time = (time.time() - start_time) / 60
        print(f"⚡ Parallel analysis completed in {parallel_time:.2f} minutes")

        self.log_time("Parallel Analysis Complete")

        if not self.all_results:
            print("❌ No valid results were generated. Cannot create summary analysis. Exiting.")
            return

        # Create summary analysis
        self._create_summary_analysis()

        # Create Excel report
        self._create_excel_report()

        # Print final summary
        self._print_final_summary()

        self.log_time("Complete Analysis Finished")

    def _create_ratio_visualizations(self, ratio_results, ratio_dir, ratio_name):
        """Create visualizations for a specific ratio."""

        # Model performance comparison for this ratio
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle(f'Model Performance for Train/Test Ratio: {ratio_name.replace("_", "/")}',
                     fontsize=16, fontweight='bold')

        model_names = list(ratio_results.keys())
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

        # Test MAPE comparison
        test_mape = [ratio_results[name]['test_metrics']['mape'] for name in model_names]
        bars = axes[0, 0].bar(model_names, test_mape, color=colors)
        axes[0, 0].set_ylabel('Test MAPE (%)')
        axes[0, 0].set_title('Test MAPE Comparison')
        axes[0, 0].tick_params(axis='x', rotation=45)

        for bar, value in zip(bars, test_mape):
            axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(test_mape)*0.01,
                           f'{value:.2f}%', ha='center', va='bottom', fontweight='bold')

        # Test R² comparison
        test_r2 = [ratio_results[name]['test_metrics']['r2'] for name in model_names]
        bars = axes[0, 1].bar(model_names, test_r2, color=colors)
        axes[0, 1].set_ylabel('Test R² Score')
        axes[0, 1].set_title('Test R² Comparison')
        axes[0, 1].tick_params(axis='x', rotation=45)

        for bar, value in zip(bars, test_r2):
            axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(test_r2)*0.01,
                           f'{value:.3f}', ha='center', va='bottom', fontweight='bold')

        # Predicted vs Actual for best model (lowest MAPE)
        best_model = min(model_names, key=lambda x: ratio_results[x]['test_metrics']['mape'])
        y_test = ratio_results[best_model]['actual_test']
        y_pred = ratio_results[best_model]['predictions']

        axes[1, 0].scatter(y_test, y_pred, alpha=0.6, color=colors[0], s=20)
        axes[1, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        axes[1, 0].set_xlabel('Actual Depth to Water (m)')
        axes[1, 0].set_ylabel('Predicted Depth to Water (m)')
        axes[1, 0].set_title(f'{best_model}: Predicted vs Actual')

        # Training vs Testing MAPE
        train_mape = [ratio_results[name]['train_metrics']['mape'] for name in model_names]
        x_pos = np.arange(len(model_names))
        width = 0.35

        axes[1, 1].bar(x_pos - width/2, train_mape, width, label='Train MAPE', color='lightblue', alpha=0.8)
        axes[1, 1].bar(x_pos + width/2, test_mape, width, label='Test MAPE', color='lightcoral', alpha=0.8)
        axes[1, 1].set_ylabel('MAPE (%)')
        axes[1, 1].set_title('Training vs Testing MAPE')
        axes[1, 1].set_xticks(x_pos)
        axes[1, 1].set_xticklabels(model_names, rotation=45)
        axes[1, 1].legend()

        plt.tight_layout()
        plt.savefig(ratio_dir / 'model_performance.png', bbox_inches='tight', facecolor='white')
        plt.close()  # Close to save memory

    def _create_summary_analysis(self):
        """Create comprehensive summary analysis across all ratios."""
        print("\n📊 Creating summary analysis across all ratios...")

        # Create summary comparison plots
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        fig.suptitle('Performance Comparison Across All Train/Test Ratios', fontsize=18, fontweight='bold')

        # Prepare data for plotting
        valid_ratio_keys = sorted(self.all_results.keys())
        ratios = [f"{key.split('_')[0]}/{key.split('_')[1]}" for key in valid_ratio_keys]
        models = list(next(iter(self.all_results.values())).keys())

        # Test MAPE across ratios
        for i, model in enumerate(models):
            test_mapes = []
            for ratio_key in valid_ratio_keys:
                test_mapes.append(self.all_results[ratio_key][model]['test_metrics']['mape'])

            axes[0, 0].plot(ratios, test_mapes, marker='o', label=model, linewidth=2, markersize=6)

        axes[0, 0].set_xlabel('Train/Test Ratio')
        axes[0, 0].set_ylabel('Test MAPE (%)')
        axes[0, 0].set_title('Test MAPE vs Train/Test Ratio')
        axes[0, 0].legend()
        axes[0, 0].tick_params(axis='x', rotation=45)
        axes[0, 0].grid(True, alpha=0.3)

        # Test R² across ratios
        for i, model in enumerate(models):
            test_r2s = []
            for ratio_key in valid_ratio_keys:
                test_r2s.append(self.all_results[ratio_key][model]['test_metrics']['r2'])

            axes[0, 1].plot(ratios, test_r2s, marker='o', label=model, linewidth=2, markersize=6)

        axes[0, 1].set_xlabel('Train/Test Ratio')
        axes[0, 1].set_ylabel('Test R²')
        axes[0, 1].set_title('Test R² vs Train/Test Ratio')
        axes[0, 1].legend()
        axes[0, 1].tick_params(axis='x', rotation=45)
        axes[0, 1].grid(True, alpha=0.3)

        # Training time across ratios
        for i, model in enumerate(models):
            train_times = []
            for ratio_key in valid_ratio_keys:
                train_times.append(self.all_results[ratio_key][model]['training_time_minutes'])

            axes[0, 2].plot(ratios, train_times, marker='o', label=model, linewidth=2, markersize=6)

        axes[0, 2].set_xlabel('Train/Test Ratio')
        axes[0, 2].set_ylabel('Training Time (minutes)')
        axes[0, 2].set_title('Training Time vs Train/Test Ratio')
        axes[0, 2].legend()
        axes[0, 2].tick_params(axis='x', rotation=45)
        axes[0, 2].grid(True, alpha=0.3)

        # Sample sizes across ratios
        train_sizes = []
        test_sizes = []
        for ratio_key in valid_ratio_keys:
            # Use first model as they all have same sample sizes
            first_model = list(self.all_results[ratio_key].keys())[0]
            train_sizes.append(self.all_results[ratio_key][first_model]['train_size'])
            test_sizes.append(self.all_results[ratio_key][first_model]['test_size'])

        axes[1, 0].bar(ratios, train_sizes, alpha=0.7, label='Training Samples', color='lightblue')
        axes[1, 0].bar(ratios, test_sizes, bottom=train_sizes, alpha=0.7, label='Testing Samples', color='lightcoral')
        axes[1, 0].set_xlabel('Train/Test Ratio')
        axes[1, 0].set_ylabel('Number of Samples')
        axes[1, 0].set_title('Sample Size Distribution')
        axes[1, 0].legend()
        axes[1, 0].tick_params(axis='x', rotation=45)

        # Best model performance heatmap
        performance_matrix = np.zeros((len(models), len(ratios)))
        for i, model in enumerate(models):
            for j, ratio_key in enumerate(valid_ratio_keys):
                performance_matrix[i, j] = self.all_results[ratio_key][model]['test_metrics']['mape']

        im = axes[1, 1].imshow(performance_matrix, cmap='RdYlBu_r', aspect='auto')
        axes[1, 1].set_xticks(range(len(ratios)))
        axes[1, 1].set_xticklabels(ratios, rotation=45)
        axes[1, 1].set_yticks(range(len(models)))
        axes[1, 1].set_yticklabels(models)
        axes[1, 1].set_title('Test MAPE Heatmap')

        # Add colorbar
        cbar = plt.colorbar(im, ax=axes[1, 1])
        cbar.set_label('Test MAPE (%)')

        # Add text annotations
        for i in range(len(models)):
            for j in range(len(ratios)):
                text = axes[1, 1].text(j, i, f'{performance_matrix[i, j]:.1f}',
                                     ha="center", va="center", color="white" if performance_matrix[i, j] > (performance_matrix.max() * 0.7) else "black", fontweight='bold')


        # Overfitting analysis (Test MAPE - Train MAPE)
        for i, model in enumerate(models):
            overfitting_scores = []
            for ratio_key in valid_ratio_keys:
                train_mape = self.all_results[ratio_key][model]['train_metrics']['mape']
                test_mape = self.all_results[ratio_key][model]['test_metrics']['mape']
                overfitting_scores.append(test_mape - train_mape)

            axes[1, 2].plot(ratios, overfitting_scores, marker='o', label=model, linewidth=2, markersize=6)

        axes[1, 2].set_xlabel('Train/Test Ratio')
        axes[1, 2].set_ylabel('Overfitting Score (Test MAPE - Train MAPE)')
        axes[1, 2].set_title('Overfitting Analysis')
        axes[1, 2].legend()
        axes[1, 2].tick_params(axis='x', rotation=45)
        axes[1, 2].grid(True, alpha=0.3)
        axes[1, 2].axhline(y=0, color='red', linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.savefig(self.main_output_dir / 'comprehensive_ratio_analysis.png',
                   bbox_inches='tight', facecolor='white', dpi=300)
        plt.show()

    def _create_excel_report(self):
        """Create comprehensive Excel report with all results."""
        print("\n📊 Creating comprehensive Excel report...")

        excel_path = self.main_output_dir / 'comprehensive_results.xlsx'
        valid_ratio_keys = sorted(self.all_results.keys())

        with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

            # Summary sheet
            summary_data = []
            for ratio_key in valid_ratio_keys:
                train_pct, test_pct = ratio_key.split('_')
                ratio_display = f"{train_pct}/{test_pct}"

                for model_name in self.all_results[ratio_key].keys():
                    result = self.all_results[ratio_key][model_name]

                    summary_data.append({
                        'Train_Test_Ratio': ratio_display,
                        'Model': model_name,
                        'Train_Size': result['train_size'],
                        'Test_Size': result['test_size'],
                        'Train_MAPE': result['train_metrics']['mape'],
                        'Test_MAPE': result['test_metrics']['mape'],
                        'Train_R2': result['train_metrics']['r2'],
                        'Test_R2': result['test_metrics']['r2'],
                        'Train_RMSE': result['train_metrics']['rmse'],
                        'Test_RMSE': result['test_metrics']['rmse'],
                        'Train_MAE': result['train_metrics']['mae'],
                        'Test_MAE': result['test_metrics']['mae'],
                        'CV_RMSE': result['cv_rmse'],
                        'Training_Time_Minutes': result['training_time_minutes'],
                        'Overfitting_Score': result['test_metrics']['mape'] - result['train_metrics']['mape']
                    })

            summary_df = pd.DataFrame(summary_data)
            summary_df.to_excel(writer, sheet_name='Summary', index=False)

            # Best performance sheet
            best_results = []
            for ratio_key in valid_ratio_keys:
                train_pct, test_pct = ratio_key.split('_')
                ratio_display = f"{train_pct}/{test_pct}"

                # Find best model for this ratio (lowest test MAPE)
                best_model = min(self.all_results[ratio_key].keys(),
                               key=lambda x: self.all_results[ratio_key][x]['test_metrics']['mape'])

                result = self.all_results[ratio_key][best_model]

                best_results.append({
                    'Train_Test_Ratio': ratio_display,
                    'Best_Model': best_model,
                    'Test_MAPE': result['test_metrics']['mape'],
                    'Test_R2': result['test_metrics']['r2'],
                    'Test_RMSE': result['test_metrics']['rmse'],
                    'Test_MAE': result['test_metrics']['mae'],
                    'Training_Time_Minutes': result['training_time_minutes'],
                    'Train_Size': result['train_size'],
                    'Test_Size': result['test_size']
                })

            best_df = pd.DataFrame(best_results)
            best_df.to_excel(writer, sheet_name='Best_Per_Ratio', index=False)

            # --- FIX IS HERE ---
            # Define the list of model names before the loop
            models = list(self.models.keys())
            
            # Individual model sheets
            for model_name in models:
                model_data = []
                for ratio_key in valid_ratio_keys:
                    train_pct, test_pct = ratio_key.split('_')
                    ratio_display = f"{train_pct}/{test_pct}"

                    if model_name in self.all_results[ratio_key]:
                        result = self.all_results[ratio_key][model_name]

                        model_data.append({
                            'Train_Test_Ratio': ratio_display,
                            'Train_Size': result['train_size'],
                            'Test_Size': result['test_size'],
                            'Train_MAPE': result['train_metrics']['mape'],
                            'Test_MAPE': result['test_metrics']['mape'],
                            'Train_R2': result['train_metrics']['r2'],
                            'Test_R2': result['test_metrics']['r2'],
                            'Train_RMSE': result['train_metrics']['rmse'],
                            'Test_RMSE': result['test_metrics']['rmse'],
                            'Train_MAE': result['train_metrics']['mae'],
                            'Test_MAE': result['test_metrics']['mae'],
                            'CV_RMSE': result['cv_rmse'],
                            'Training_Time_Minutes': result['training_time_minutes']
                        })

                model_df = pd.DataFrame(model_data)
                sheet_name = model_name.replace(' ', '_')[:31]  # Excel sheet name limit
                model_df.to_excel(writer, sheet_name=sheet_name, index=False)

            # Feature importance sheet (using one ratio as example, e.g., the first valid one)
            first_ratio_key = valid_ratio_keys[0]
            feature_importance_data = []

            for model_name in self.all_results[first_ratio_key].keys():
                model = self.all_results[first_ratio_key][model_name]['model']
                if hasattr(model, 'feature_importances_'):
                    importances = model.feature_importances_
                    for i, importance in enumerate(importances):
                        feature_importance_data.append({
                            'Model': model_name,
                            'Feature': self.feature_cols[i],
                            'Importance': importance
                        })

            if feature_importance_data:
                fi_df = pd.DataFrame(feature_importance_data).sort_values(by=['Model', 'Importance'], ascending=[True, False])
                fi_df.to_excel(writer, sheet_name='Feature_Importance', index=False)

        print(f"✅ Excel report saved to: {excel_path}")

    def _print_final_summary(self):
        """Print comprehensive final summary."""
        print("\n" + "="*100)
        print("🎉 COMPREHENSIVE ANALYSIS COMPLETE - MULTI-RATIO EVALUATION")
        print("="*100)

        # Find overall best performing combination
        best_combo = None
        best_mape = float('inf')

        for ratio_key in self.all_results.keys():
            train_pct, test_pct = ratio_key.split('_')
            ratio_display = f"{train_pct}/{test_pct}"

            for model_name in self.all_results[ratio_key].keys():
                test_mape = self.all_results[ratio_key][model_name]['test_metrics']['mape']
                if test_mape < best_mape:
                    best_mape = test_mape
                    best_combo = (ratio_display, model_name)

        if best_combo:
            print(f"🏆 BEST OVERALL PERFORMANCE:")
            print(f"   Ratio: {best_combo[0]} | Model: {best_combo[1]} | Test MAPE: {best_mape:.2f}%")
        else:
            print("🏆 No best model could be determined.")

        # Summary statistics
        print(f"\n📊 ANALYSIS SUMMARY:")
        print(f"   Total Ratios Tested: {len(self.test_ratios)}")
        print(f"   Models per Ratio: {len(self.models)}")
        print(f"   Total Successful Model Runs: {sum(len(v) for v in self.all_results.values())}")
        print(f"   Results Saved to: {self.main_output_dir}")

        # Performance trends
        print(f"\n📈 KEY INSIGHTS:")

        # Best ratio for each model
        for model_name in self.models.keys():
            best_ratio_for_model = None
            best_mape_for_model = float('inf')

            for ratio_key in self.all_results.keys():
                if model_name in self.all_results[ratio_key]:
                    test_mape = self.all_results[ratio_key][model_name]['test_metrics']['mape']
                    if test_mape < best_mape_for_model:
                        best_mape_for_model = test_mape
                        best_ratio_for_model = ratio_key.replace('_', '/')
            if best_ratio_for_model:
                print(f"   • {model_name}: Best at {best_ratio_for_model} ratio (MAPE: {best_mape_for_model:.2f}%)")
            else:
                 print(f"   • {model_name}: No valid results found.")

        # Timing summary
        if self.timing_log:
            total_time = self.timing_log[-1]['elapsed_minutes']
            print(f"\n⏰ TOTAL RUNTIME: {total_time:.2f} minutes")

        print("="*100)

def main():
    """Main execution function."""
    # Update this path to your actual file location
    file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"

    # Initialize and run analysis
    predictor = GroundwaterPredictorMultiRatio(file_path)
    predictor.load_and_preprocess_data()

    # Run parallel analysis with all available CPU cores
    predictor.run_parallel_analysis(n_jobs=-1)

if __name__ == "__main__":
    # This check ensures that the main function runs only when the script is executed directly
    # and not when imported as a module. It's crucial for multiprocessing on some platforms.
    main()